Линейный слой - это функция, принимает вектор одного размера, превращает в вектор другого размера.

- x - входной вектор (фичи)  
- W - матрица весов (Weights), главные параметры слоя  
- b - вектор смещения (Bias), одно число для каждого выходного нейрона  
- y - выходной вектор  

---

#### Линейный слой со смещением (Dense, Linear, Fully Connected)
  
>y = xW**T + b

Линия/плоскость может двигаться свободно.  

- Когда применять? Классические задачи, финальные слои (регрессия, классификация).

Пример:  
x = [площадь, удаленность_от_центра]  
weight1 = веса площади  
weight2 = веса удаленности от центра  
b = одно смещение вверх или вниз = базовое значение переменной, даже если фичи = 0  
y = цена  

---

#### Линейный слой без смещения  
>y = xW**T  

Линия/плоскость жестко привязана к центру (0,0)  

- Когда применять? Трансформеры (Q, K, V матрицы), слои перед Batch/Layer Norm

In [ ]:
from torch.nn import nn

dense_without_bias = nn.Linear(in_features, out_features, bias=False)

На первый взгляд кажется, что убирая bias, мы ухудшаем гибкость модели. 
 
---
 
Три важнейших кейса в современном Deep Learning (особенно в LLM и трансформерах), где bias смещение **намеренно отключают:**

>Слои Attension (Внимание) в трансформерах (GPT, BERT и др)   

В механизме Self-Attention входные векторы умножаются на матрицы W_Q, W_K, W_V (Query, Key, Value). Здесь bias часто отключают (bias=False), потому что нам важна только относительная геометрическая связь (угол и проекция) между векторами слов. **Смещение может исказить** эти чистые пространственные отношения.

>Перед слоями нормализации (Batch Normalization / Layer Normalization)  

Если сразу после линейного слоя у тебя стоит, например, Batch Norm, он первым делом центрирует данные — то есть вычитает среднее значение по батчу. Математически, Layer/Batch Norm просто "уничтожает" работу, сделанную коэффициентом bias. Зачем тратить память и ресурсы на обучение параметров, которые следующий шаг всё равно обнулит?

>Экономия памяти и вычислений

В гигантских моделях на миллиарды параметров каждый сохраненный байт на счету. Убрав bias там, где он не влияет на качество, инженеры экономят память видеокарты (VRAM) и ускоряют вычисления.

# База

### 1. Производная - коэффициент масштабирования

>y = f(x)  

Если y = 3x, то проивзодная равна 3.   
Это значит, если x += 1, то y += 3.

In [ ]:
def predict(x):
    return 3 * x

# Если x увеличился на 0.01, то y увеличился на 0.03

### 2. Что такое Loss (функция потерь)?

Loss, обозначается как L.   

Принимает предсказание сети Y и правильный ответ Y_true, а на выходе выдает одно число — **штраф за ошибку**  
>Наша главная цель — сделать этот штраф L как можно меньше (ближе к нулю).

Для этого нам нужно изменить веса сети W. 
>Как узнать в какую сторону их крутить?

Нужно найти производную лосса по весам: dL / dW.  

 «Если мы изменим вес W на 0.001, уменьшится лосс или увеличится?»

### 3. Правило цепочки (Chain Rule)

В нейросетях данные идут последовательно, как по конвейеру (или как цепочка вызовов функций в коде): 
 
>X -> Линейный слой -> Y -> Функция потерь -> L

In [ ]:
y = forward_linear(x, w) # линейный слой
loss = compute_loss(y, y_true) # функция потерь

Когда **лосс** посчитан, начинается **Backpropagation** - обратное распространение ошибки.  
Мы идем справа налево.

1. Считаем как **лосс** изменился из-за выхода слоя: dL / dY
2. Нам нужно узнать, как **лосс** изменился из-за весов W: dL / dW

Математика (правило цепочки) говорит: чтобы узнать, как крайний правый элемент L зависит от крайнего левого W, нужно перемножить промежуточные производные: dL / dW = (dL / dY) * (dY / dW)

Это и есть суть бэкпропа: верхний слой дает нам готовый dY = dL / dY, линейный слой сам считает свою локальную производную dY / dW и умножает их друг на друга.



### 4. Матрицы

В реальном ML мы не передаем по одному числу — это слишком медленно для процессоров. Мы передаем данные пачками (батчами).

- X — это таблица (матрица), где строки — это объекты (например, 32 квартиры), а столбцы — их признаки (площадь, этаж).
- W — это матрица коэффициентов.
- Y = X ⋅ W — это тоже матрица (предсказания для всех 32 квартир).

Когда мы считаем производные для матриц, обычное умножение превращается в **матричное умножение.** И тут возникает суровая проблема: порядок множителей имеет значение. В обычной математике 2 ⋅ 3 = 3 ⋅ 2, а в матрицах **A ⋅ B ≠ B ⋅ A**. Более того, их часто вообще нельзя умножить, если у них **не совпадают** внутренние размерности.

Поэтому можно перевернуть (транспонировать) X, превратив его в X^T, чтобы матрицы вообще смогли перемножиться по правилам линейной алгебры и дать на выходе сетку правильного размера (такого же, как сами веса W).

>смысл (**«умножить ошибку на вход»** - для весов и **«умножить ошибку на вес»** - для входа)

Мы напишем класс, в котором будут два метода: forward (расчет предсказания) и backward (расчет градиентов по тем самым формулам, что мы вывели).

#### Шаг 1: Код слоя на чистом Python

Почему именно Y = X ⋅ W? (Линейная комбинация) - Это самый простой способ смешать информацию.

Анализ размерностей (тензорная геометрия)

Комментарий в коде очень важен для понимания того, как данные меняют форму:  
(N x D) @ (D x M) -> (N x M)

- \(N\) **(Batch Size):** Количество примеров (например, картинок или текстов), которые сеть обрабатывает одновременно за один шаг.
- \(D\) **(Input Features):** Количество признаков на входе. Например, если на вход подается картинка 28x28 пикселей в виде вектора, то D = 784.
- \(M\) **(Output Features):** Количество нейронов в текущем слое (сколько признаков мы хотим получить на выходе).
- **Правило умножения:** Чтобы матрицы можно было умножить, внутренняя размерность должна   
совпадать (D и D). На выходе мы получаем матрицу размером (N x M) — новые признаки для   
каждого из N примеров.

In [ ]:
import numpy as np

class PureLinearLayerWithoutBias:
    def __init__(self, in_features, out_features):
        # init матрицу весов случайными числами
        # Размерность D x M
        self.W = np.random.randn(in_features, out_features) * 0.01
        # Вход X во время forward pass, понадобиться на backward pass для вычисления градиентов
        self.X = None
    
    def forward(self, X):
        # Сохранение входа для backpropagation
        self.X = X
        # Классическое матричное умножение (dot product)
        Y = self.X @ self.W
        # Матрица Y передается дальше по цепочке
        return Y
    
    def backward(self, dY):
        # dY — это градиент от лосса

        # 1. Считаем градиент по весам W
        dW = self.X.T @ dY
        
        # 2. Считаем градиент по входу X
        dX = dY @ self.W.T
        
        # Сохраняем градиент весов внутри слоя, чтобы оптимизатор (например, SGD) 
        # потом мог обновить веса: W = W - learning_rate * dW
        self.dW = dW
        
        # Возвращаем dX обратно, чтобы предыдущий слой тоже мог посчитать свои градиенты
        return dX


#### Шаг 2: Эмуляция работы (Запуск конвейера)  

- Батч из N=2 объектов (например, две квартиры).
- У каждого объекта D=3 признака (площадь, комнат, удаленность).
- На выходе нам нужно получить M=4 скрытых признака.

In [ ]:
# Создаем наш слой: 3 входа, 4 выхода
layer = PureLinearLayerWithoutBias(in_features=3, out_features=4)

# 1. FORWARD PASS
# Генерируем фейковый батч данных X размера (2 x 3)
X = np.array([[1.0, 2.0, 3.0],
              [4.0, 5.0, 6.0]])

# Считаем выход Y размера (2 x 4)
Y = layer.forward(X)
print("Размерность выхода Y:", Y.shape)  # Выведет: (2, 4)


# 2. BACKWARD PASS
# Представим, что лосс посчитал ошибку и прислал нам dY размера (2 x 4)
dY = np.array([[0.1, -0.2, 0.3, 0.0],
               [0.5, 0.1, -0.1, 0.2]])

# Пробрасываем градиент обратно
dX = layer.backward(dY)

print("Размерность dX (градиент по входу):", dX.shape) # Выведет: (2, 3) — как у X!
print("Размерность dW (градиент по весам):", layer.dW.shape) # Выведет: (3, 4) — как у W!


Подведем итог для разработчика

В бэкенде или фронтенде данные просто текут в одну сторону. В ML/DL код всегда работает в две стороны:

1. В `forward` данные превращаются в предсказания. Слой обязан **запомнить свой вход `X`**, потому что он — часть локальной производной.
2. В `backward` мы берем пришедшую ошибку `dY`, перемножаем её с сохраненным `X` (для обновления весов) и с матрицей весов `W` (чтобы передать ошибку дальше по цепочке).

Ура, ты своими руками разобрал базу, на которой строятся абсолютно все библиотеки глубокого обучения (PyTorch, TensorFlow, Jax)!

---